# Tariff example usage
This notebook demonstrates how to use the `Tariff` class to calculate energy costs based on a given tariff structure. We will create a sample tariff and then use it to compute the cost of energy consumption over a specified period.

### 1. Creating an index
Most tariffs are based on one or more indexes. For example, a common structure is to have a fixed cost component and a variable cost component that depends on the day-ahead electricity price. Let's create an index for the day-ahead price using the `EntsoeDayAheadIndex`:

In [1]:
from os import environ

from energy_cost.index import EntsoeDayAheadIndex, Index

Index.register("Belpex15min", EntsoeDayAheadIndex(country_code="BE", api_key=environ["ENTSOE_API_KEY"]))

### 2. Defining a tariff
The easiest way to define a tariff is to specify it in a YAML file. This allows for a clear separation of the tariff definition from the code that uses it. You can find some examples in `data/tariffs/`. 

A simple dynamic tariff might look like this:

In [2]:
from helpers import display_yaml

display_yaml("../data/tariffs/foo/dynamic_tariff.yml")

```yaml
supplier: "Foo" # The energy supplier providing the tariff
product: "Bar" # The specific product or plan offered by the supplier
defaults: # Default price formulas used for this tariff
  - start: 2025-01-01T00:00:00+01 # The start date of validity for the formula, in ISO 8601 format
    formula:
      constant_cost: 10.0 # The fixed cost component in €/MWh
      variable_costs: # The variable cost component depends on one or more indexes, which are multiplied by a scalar and summed to calculate the cost
      - index: Belpex15min # The index used for the variable cost, in €/MWh
        scalar: 1.05 # The scalar applied to the index to calculate the variable cost
  - start: 2026-01-01T00:00:00+01 # A new formula that becomes valid from 2026-01-01, overriding the previous one
    formula:
      constant_cost: 12.0
      variable_costs:
      - index: Belpex15min
        scalar: 1.10
```

In [3]:
from energy_cost.tariff import Tariff

tariff = Tariff.from_yaml("../data/tariffs/foo/dynamic_tariff.yml")
tariff

Tariff(supplier='Foo', product='Bar', defaults={<PowerDirection.CONSUMPTION: 'consumption'>: {<CostType.ENERGY: 'energy'>: [TimedPriceFormula(start=datetime.datetime(2025, 1, 1, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))), formula=PriceFormula(constant_cost=10.0, variable_costs=[IndexAdder(index='Belpex15min', scalar=1.05)])), TimedPriceFormula(start=datetime.datetime(2026, 1, 1, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))), formula=PriceFormula(constant_cost=12.0, variable_costs=[IndexAdder(index='Belpex15min', scalar=1.1)]))]}}, by_meter_type={})

### 3. Calculating cost per energy consumption in a given period
Once we have defined our tariff, we can use it to calculate the cost of energy consumption over a specified period. This is done by calling the `get_cost` method of the `Tariff` class, which returns a DataFrame with the cost for each time step in the given period.

In [4]:
import datetime as dt

start = dt.datetime.fromisoformat("2026-03-08 00:00:00+01:00")
end = dt.datetime.fromisoformat("2026-03-10 00:00:00+01:00")
tariff.get_cost(start, end)

,timestamp,energy,total
0,2026-03-08 00:00:00+01:00,162.392,162.392
1,2026-03-08 00:15:00+01:00,153.471,153.471
2,2026-03-08 00:30:00+01:00,148.840,148.840
3,2026-03-08 00:45:00+01:00,143.758,143.758
4,2026-03-08 01:00:00+01:00,153.152,153.152
...,...,...,...
187,2026-03-09 22:45:00+01:00,156.144,156.144
188,2026-03-09 23:00:00+01:00,166.374,166.374
189,2026-03-09 23:15:00+01:00,160.665,160.665
190,2026-03-09 23:30:00+01:00,154.472,154.472


### 4. Specifying injection tariffs
The `Tariff` class also supports injection tariffs, which can be used to calculate the revenue from feeding energy back into the grid. This is done by defining a separate price formula list for injection calculations in the tariff definition. The `get_cost` method can then be used to calculate the revenue from injections over a specified period, similar to how it calculates costs for consumption.

In [6]:
display_yaml("../data/tariffs/foo/injection_tariff.yml")

```yaml
supplier: foo
product: dynamic tariff
defaults:
  injection: # specifying a separate price formula list for injection calculations
  - start: 2025-01-01T00:00:00+01
    formula:
      constant_cost: -14.56
      variable_costs:
      - index: Belpex15min
        scalar: 0.9876
  consumption: # the consumption price formulas now also need to be named explicitly, as they are no longer the default
  - start: 2025-01-01T00:00:00+01
    formula:
      constant_cost: 12.34
      variable_costs:
      - index: Belpex15min
        scalar: 1.23
```

In [7]:
from energy_cost.tariff import PowerDirection

tariff = Tariff.from_yaml("../data/tariffs/foo/injection_tariff.yml")
tariff.get_cost(start, end, direction=PowerDirection.INJECTION)

,timestamp,energy,total
0,2026-03-08 00:00:00+01:00,120.464672,120.464672
1,2026-03-08 00:15:00+01:00,112.455236,112.455236
2,2026-03-08 00:30:00+01:00,108.297440,108.297440
3,2026-03-08 00:45:00+01:00,103.734728,103.734728
4,2026-03-08 01:00:00+01:00,112.168832,112.168832
...,...,...,...
187,2026-03-09 22:45:00+01:00,114.855104,114.855104
188,2026-03-09 23:00:00+01:00,124.039784,124.039784
189,2026-03-09 23:15:00+01:00,118.914140,118.914140
190,2026-03-09 23:30:00+01:00,113.353952,113.353952


### 4. Specifying meter types
The `Tariff` class also supports different prices per meter type (e.g.Time of use meters, single rate meters, etc.). This can be done by defining separate price formula lists for each meter type in the tariff definition. The `get_cost` method can then be used to calculate costs for each meter type separately.

Price formulas specified in defaults are applied to all meter types, but they can be overridden by formulas specified for a specific meter type. This allows for a flexible tariff structure that can accommodate various pricing schemes based on meter types.

In [10]:
display_yaml("../data/tariffs/foo/metered_tariff.yml")

```yaml
supplier: foo
product: dynamic tariff
defaults:
  injection: # the injection price formulas are specified in defaults, which means they apply to all meter types unless overridden
  - start: 2025-01-01T00:00:00+01
    formula:
      constant_cost: -14.56
      variable_costs:
      - index: Belpex15min
        scalar: 0.9876
by_meter_type: # the consumption price formulas are now specified under by_meter_type, which means they can be different for each meter type
  tou_peak:
    consumption:
    - start: 2025-01-01T00:00:00+01
      formula:
        constant_cost: 23.45
        variable_costs:
        - index: Belpex15min
          scalar: 2.34
  tou_off_peak:
    consumption:
    - start: 2025-01-01T00:00:00+01
      formula:
        constant_cost: 10.0
        variable_costs:
        - index: Belpex15min
          scalar: 0.56

```

In [11]:
from energy_cost.tariff import MeterType

tariff = Tariff.from_yaml("../data/tariffs/foo/metered_tariff.yml")
tariff.get_cost(start, end, meter_type=MeterType.TOU_PEAK)

,timestamp,energy,total
0,2026-03-08 00:00:00+01:00,343.3748,343.3748
1,2026-03-08 00:15:00+01:00,324.3974,324.3974
2,2026-03-08 00:30:00+01:00,314.5460,314.5460
3,2026-03-08 00:45:00+01:00,303.7352,303.7352
4,2026-03-08 01:00:00+01:00,323.7188,323.7188
...,...,...,...
187,2026-03-09 22:45:00+01:00,330.0836,330.0836
188,2026-03-09 23:00:00+01:00,351.8456,351.8456
189,2026-03-09 23:15:00+01:00,339.7010,339.7010
190,2026-03-09 23:30:00+01:00,326.5268,326.5268


### 5. complex cost components
Some regions allow explicitly listing separate cost components for the consumption based cost in the tariff definition. This can include price formulas for buying renewable energy certificates.

These can be specified next to the default `energy` cost components, allowing for a more granular and flexible tariff structure.

In [ ]:
display_yaml("../data/tariffs/foo/cost_types_tariff.yml")

```yaml
supplier: foo
product: dynamic tariff
defaults:
  injection: # no cost component specified, so these are assumed to apply to the energy injection cost
  - start: 2025-01-01T00:00:00+01
    formula:
      constant_cost: -14.56
      variable_costs:
      - index: Belpex15min
        scalar: 0.9876
  consumption:
    renewable_certificates: # specify the cost component this price formula applies to, in this case the cost of buying renewable energy certificates
    - start: 2025-01-01T00:00:00+01
      formula:
        constant_cost: 12.0
    chp_certificates: # we also support a separate cost component for the cost of buying combined heat and power certificates
    - start: 2025-01-01T00:00:00+01
      formula:
        constant_cost: 4.56
    energy: # the default cost component, the name can be omitted if there are no other cost components
    - start: 2025-01-01T00:00:00+01
      formula:
        constant_cost: 12.34
        variable_costs:
        - index: Belpex15min
          scalar: 1.23
    - start: 2026-01-01T00:00:00+01
      formula:
        constant_cost: 14.56
        variable_costs:
        - index: Belpex15min
          scalar: 1.23
  
```

In [ ]:
tariff = Tariff.from_yaml("../data/tariffs/foo/cost_types_tariff.yml")
tariff.get_cost(start, end)